### Dataset Generation

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 120

data = {
    "record_id": [f"REC{i+1:03d}" for i in range(n)],
    "age": np.random.randint(18, 70, n),
    "weight": np.random.randint(45, 100, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "smoking_status": np.random.choice(
        ["Smoker", "Non-Smoker", "Former Smoker"], n, p=[0.35, 0.45, 0.20]
    ),
    "exercise_frequency": np.random.choice(
        ["Daily", "Weekly", "Rarely", "Never"], n, p=[0.3, 0.35, 0.2, 0.15]
    ),
}

df = pd.DataFrame(data)

# Derived fields
df["bmi"] = np.round(df["weight"] / ((np.random.uniform(1.5, 1.8, n)) ** 2), 1)
df["blood_pressure"] = np.random.normal(130, 15, n)
df["cholesterol_level"] = np.random.normal(210, 30, n)
df["glucose_level"] = np.random.normal(110, 25, n)

# Diseases (probability-based)
df["diabetes"] = np.where((df["glucose_level"] > 126) | (df["bmi"] > 30), True, False)
df["hypertension"] = np.where(df["blood_pressure"] > 140, True, False)

# Age groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

# Dates
df["visit_date"] = pd.date_range(start="2023-01-01", periods=n, freq="D")

df.to_csv("health_records.csv", index=False)

## Part A - Theoretical Foundation (Short Notes & Explanation):

#### 1. Inferential Statistics
Inferential statistics is a branch of statistics that uses sample data to draw conclusions or make predictions about a larger population. It helps in estimating parameters, testing hypotheses, and determining relationships between variables.

#### 2. Hypothesis Testing and Its Components
Hypothesis testing is a statistical method used to make decisions about a population based on sample data. Its main components include the null hypothesis (H₀), alternative hypothesis (H₁), significance level (α), test statistic, critical value, and p-value.

#### 3. Confidence Interval and Critical Value
A confidence interval is a range of values within which the true population parameter is expected to lie with a certain confidence level. The critical value is a threshold obtained from a statistical distribution that determines whether the null hypothesis should be rejected.

#### 4. p-value
The p-value is the probability of obtaining results as extreme as the observed data assuming the null hypothesis is true. A smaller p-value indicates stronger evidence against the null hypothesis.

#### 5. Type I and Type II Errors
A Type I error occurs when a true null hypothesis is incorrectly rejected. A Type II error occurs when a false null hypothesis is not rejected.

#### 6. z-test, t-test, Chi-square test, and ANOVA
A z-test is used to compare means when the sample size is large and population variance is known. A t-test is used to compare means when the sample size is small or population variance is unknown. A chi-square test examines the association between categorical variables, while ANOVA compares means across three or more groups.

#### 7. Covariance
Covariance measures how two variables change together and indicates the direction of their linear relationship. A positive covariance means both variables increase together, while a negative covariance means one increases as the other decreases.

#### 8. Correlation
Correlation measures the strength and direction of the linear relationship between two variables and is standardized between -1 and +1. A value close to +1 or -1 indicates a strong relationship, while a value near 0 indicates a weak or no linear relationship.

## Part B-Data Analysis & Testing Tasks:

**1. Hypothesis Formulation**

Hypothesis 1 (Chi-Square)
- H₀: Smoking status has no association with diabetes.
- H₁: Smoking status is associated with diabetes.

Hypothesis 2 (t-test)
- H₀: Mean BMI of diabetic and non-diabetic individuals is equal.
- H₁: Mean BMI of diabetic and non-diabetic individuals is different.

In [12]:
# 2. Confidence Interval (Age & Weight)

from scipy import stats

confidence = 0.95

# Age CI
age_mean = df["age"].mean()
age_sem = stats.sem(df["age"])
age_ci = stats.t.interval(confidence, len(df)-1, age_mean, age_sem)

# Weight CI
weight_mean = df["weight"].mean()
weight_sem = stats.sem(df["weight"])
weight_ci = stats.t.interval(confidence, len(df)-1, weight_mean, weight_sem)

age_ci, weight_ci

((np.float64(40.08962543351388), np.float64(45.560374566486125)),
 (np.float64(70.43937750946813), np.float64(76.22728915719853)))

In [13]:
# 3. Critical Value & p-Value (t-test)

critical_value = stats.t.ppf(1 - 0.05/2, df=len(df)-2)
critical_value

np.float64(1.9802722492407059)

In [14]:
# 4. t-Test (BMI vs Diabetes)

bmi_diabetic = df[df["diabetes"] == True]["bmi"]
bmi_non_diabetic = df[df["diabetes"] == False]["bmi"]

t_stat, p_value = stats.ttest_ind(bmi_diabetic, bmi_non_diabetic, equal_var=False)
t_stat, p_value

if p_value < 0.05:
    print("Reject H0")
    print("BMI differs significantly between diabetic and non-diabetic individuals")
else:
    print("Accept H0")
    print("Mean BMI of diabetic and non-diabetic individuals is equal.")

Reject H0
BMI differs significantly between diabetic and non-diabetic individuals


In [15]:
# 5. Chi-Square Test (Smoking vs Diabetes)

contingency = pd.crosstab(df["smoking_status"], df["diabetes"])

chi2, p, dof, expected = stats.chi2_contingency(contingency)

chi2, p

if p_value < 0.05:
    print("Reject H0")
    print("Smoking status is associated with diabetes.")
else:
    print("Accept H0")
    print("Smoking status has no association with diabetes.")

Reject H0
Smoking status is associated with diabetes.


In [16]:
# 6. ANOVA (Age Group vs Glucose Level)

groups = [
    df[df["age_group"] == grp]["glucose_level"]
    for grp in df["age_group"].dropna().unique()
]

f_stat, p_val = stats.f_oneway(*groups)
f_stat, p_val

if p_value < 0.05:
    print("Reject H0")
    print("Mean glucose levels differ significantly across age groups")
else:
    print("Accept H0")
    print("Mean glucose levels don't differ across age groups")

Reject H0
Mean glucose levels differ significantly across age groups


In [17]:
# 7. Covariance & Correlation (Age vs BMI)

covariance = np.cov(df["age"], df["bmi"])[0, 1]
correlation = np.corrcoef(df["age"], df["bmi"])[0, 1]

covariance, correlation

if covariance > 0:
    print("BMI increases with age gap")
else:
    print("BMI decreases with age gap")

BMI increases with age gap


**8. Final Summary**

- Smoking habit shows a statistically significant association with diabetes.

- BMI differs significantly between diabetic and non-diabetic individuals.

- Age groups demonstrate varying glucose levels, confirmed by ANOVA.

- Age and BMI exhibit a positive correlation.

- Hence, lifestyle and physiological factors significantly affect disease occurrence.